In [1]:
import pandas as pd

In [2]:
train = pd.read_csv(r"D:\Databases\churn-prediction-project\data\raw\train.csv")
members = pd.read_csv(r"D:\Databases\churn-prediction-project\data\raw\members_v3.csv")
transaction = pd.read_csv(r"D:\Databases\churn-prediction-project\data\raw\transactions.csv")
logs = pd.read_csv(r"D:\Databases\churn-prediction-project\data\processed\aggregated_logs.csv")

In [6]:
transaction = transaction.sort_values(by = ['msno', 'transaction_date'])
latest_tx = transaction.drop_duplicates(subset = ['msno'], keep = 'last').copy()

In [7]:
tx_count = transaction.groupby('msno').size().reset_index(name = 'historical_tx_count')
latest_tx = pd.merge(latest_tx, tx_count, on = 'msno', how = 'left')

In [8]:
master_df = pd.merge(train, members, on = 'msno', how = 'left')
master_df = pd.merge(master_df, latest_tx, on = 'msno', how = 'left')
master_df = pd.merge(master_df, logs, on = 'msno', how = 'left')

In [14]:
master_df['completion_ratio'] = master_df['total_songs_finished'] / (master_df['total_songs_finished'] + master_df['total_songs_skipped'] + 1e-5) 

In [17]:
logs_col = ['total_songs_skipped', 'total_songs_finished', 'total_time_listened_in_secs', 'completion_ratio']
master_df[logs_col] = master_df[logs_col].fillna(0)

In [ ]:
master_df.to_csv(r'D:\Databases\churn-prediction-project\data\processed\final_training_data.csv', index = False)

In [19]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 992931 entries, 0 to 992930
Data columns (total 21 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   msno                         992931 non-null  str    
 1   is_churn                     992931 non-null  int64  
 2   city                         877161 non-null  float64
 3   bd                           877161 non-null  float64
 4   gender                       391692 non-null  str    
 5   registered_via               877161 non-null  float64
 6   registration_init_time       877161 non-null  float64
 7   payment_method_id            992931 non-null  int64  
 8   payment_plan_days            992931 non-null  int64  
 9   plan_list_price              992931 non-null  int64  
 10  actual_amount_paid           992931 non-null  int64  
 11  is_auto_renew                992931 non-null  int64  
 12  transaction_date             992931 non-null  int64  
 13  membership